# Data Center Policy Restriction Tracker

Exploration notebook over the local SQLite database.

Run `python scripts/build_db.py` first if `data/tracker.db` does not exist.

Source: Moratorium Nation (Bommarito, 2026), CC-BY-4.0.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import pandas as pd
from scripts.queries import (summary, in_force_by_state, jurisdiction_history,
                            flagged_for_verification, timeline_by_month, no_end_date)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

## Headline counts

Data-center-relevant instruments only. Pass `data_center_only=False` for the full inventory
including solar, wind, battery storage, and crypto mining.

In [ ]:
summary()

## Where restrictions are actually in force

`in_force` means `enacted_status` is active or extended. Everything else is pending,
expired, replaced, or rescinded.

In [ ]:
in_force_by_state().head(15)

## One jurisdiction's full history

Partial name match. Add a state abbreviation when the name is common.

In [ ]:
jurisdiction_history('Loudoun')

In [ ]:
jurisdiction_history('Columbus', 'OH')

## Adoption over time

The cumulative column is the line to show a client. The monthly column is the one that
gets misread, because the most recent month is almost always incomplete.

In [ ]:
t = timeline_by_month()
t.tail(12)

In [ ]:
ax = t.set_index('month')['cumulative'].plot(figsize=(11,4),
        title='Cumulative data center moratorium instruments')
ax.set_xlabel('')
ax.set_ylabel('instruments')

## Reconciliation candidates

Rows the upstream dataset flagged for verification, or that carry few citations.
This is the working list for Phase 2: anything here needs the primary document pulled
before it goes in front of a client.

In [ ]:
fv = flagged_for_verification()
print(len(fv), 'flagged rows')
fv.head(20)

## Data quality check: missing end dates

The usual framing is that these pauses are temporary and self-expiring. Check how many
rows actually carry a parsed end date before repeating that claim.

In [ ]:
ne = no_end_date()
print(f'{len(ne)} rows lack a parsed end date')
ne.head(15)

## Scratch

Raw SQL when the helpers do not cover it.

In [ ]:
import sqlite3
con = sqlite3.connect('data/tracker.db')
pd.read_sql_query('''
    SELECT jurisdiction_type, COUNT(*) AS n
    FROM v_data_center
    GROUP BY jurisdiction_type
    ORDER BY n DESC
''', con)